# Cryoablation dosimetry: thermal model and dose-response

Reproduces the quantities reported in the manuscript: the finite-difference
thermal model checked against measured thermocouples, the hierarchical
dose-response fits and the likelihood comparison behind them, the dose
metrics, the multi-cycle survival test, the lung-adapted ice-ball geometry
and its agreement with the patient cohort.

Inputs are in `data/` and in the thermocouple directories. Running the whole
notebook takes roughly 20 minutes; the sampling cells account for most of it.
The input sensitivity analysis is slower and lives in
`supplementary/input_sensitivity.py`.

In [1]:
import numpy as np
import pandas as pd

import cryo_thermal as ct
import dose_response as dr

pd.set_option('display.width', 140)
print('NUTS backend:', dr.NUTS_BACKEND or 'pymc default')

NUTS backend: nutpie


## 1. Thermal model against measured thermocouples

One offset (dT, the gap between the boundary thermocouple and the true probe
surface) is fitted per experiment; every other property is fixed at its
literature value. R2 and mean absolute error are reported at the validation
thermocouples, which are not used in the fit.

In [2]:
EXPS = {e['id']: e for e in ct.build_experiments('.')}

# Boundary condition as used throughout the study: pure conduction, and a
# zero-flux probe boundary after the nadir to represent passive rewarming.
W_CONV, NEUMANN_THAW = 0.0, True

fits = []
for exp_id, exp in EXPS.items():
    res = ct.run_experiment(exp, W_conv=W_CONV, neumann_thaw=NEUMANN_THAW)
    fits.append(dict(experiment=exp_id, stage=exp['stage'], dT_C=res['dT'],
                     R2_all=res['R2_all'], R2_val=res['R2_val'],
                     MAE_val_C=res['MAE_val']))
thermal = pd.DataFrame(fits)
print(thermal.to_string(index=False, float_format=lambda v: f'{v:9.3f}'))

C:\Users\arman\Documents\GitHub\Crystal 2\crystal-dosimetry-code\cryo_thermal.py:564: RuntimeWarning: All-NaN slice encountered
  T_bc_raw = np.nanmin(temps[:, bc_cols], axis=1)


        experiment      stage      dT_C    R2_all    R2_val  MAE_val_C
 celsio_A549_N1_n1     celsio   -63.693     0.842     0.769      3.564
 celsio_A549_N1_n3     celsio   -61.263     0.556     0.587      5.457
 celsio_A549_N1_n4     celsio   -75.844     0.690     0.491      5.949
 celsio_A549_N2_n1     celsio   -63.250     0.812     0.811      3.393
 celsio_A549_N2_n2     celsio   -63.737     0.624     0.745      4.351
 celsio_A549_N2_n3     celsio   -68.956     0.607     0.618      5.498
 celsio_A549_N4_n1     celsio   -73.352     0.720     0.597      6.016
 celsio_A549_N4_n2     celsio   -69.243     0.746     0.610      5.890
 celsio_A549_N4_n3     celsio   -67.665     0.721     0.660      6.057
celsio_CALU1_N1_n1     celsio   -54.963     0.889     0.809      3.437
celsio_CALU1_N1_n2     celsio   -49.607     0.910     0.882      3.108
celsio_CALU1_N1_n3     celsio   -54.452     0.890     0.866      3.053
celsio_CALU1_N2_n1     celsio   -61.844     0.726     0.475      4.686
celsio

In [3]:
# Per-stage summary. R2 is unstable where the validation thermocouples cool
# only a few degrees and there is little variance to explain, so the median is
# reported alongside the mean, and absolute error is the more stable metric.
summary = (thermal.groupby('stage')
           .agg(n=('R2_val', 'size'),
                dT_mean=('dT_C', 'mean'), dT_sd=('dT_C', 'std'),
                R2_mean=('R2_val', 'mean'), R2_median=('R2_val', 'median'),
                R2_sd=('R2_val', 'std'), MAE_mean=('MAE_val_C', 'mean'),
                MAE_sd=('MAE_val_C', 'std')))
print(summary.to_string(float_format=lambda v: f'{v:8.3f}'))
print('negative R2_val:', int((thermal.R2_val < 0).sum()), 'of', len(thermal))

             n  dT_mean    dT_sd  R2_mean  R2_median    R2_sd  MAE_mean   MAE_sd
stage                                                                           
celsio      27  -63.189    7.273    0.544      0.778    0.886     4.319    2.179
clinical     9  -12.979   12.224    0.780      0.871    0.200     6.531    3.612
crystal_3d   3   -2.287    2.468    0.849      0.816    0.098     1.741    0.351
ice2x6       6   -4.761   13.974    0.894      0.898    0.019     4.859    0.830
negative R2_val: 2 of 45


## 2. Experimental configuration

Thermocouple positions in both coordinate conventions, the freeze and thaw
timings recovered from the acquisition records, and the coldest temperature
each configuration actually measured out of sample.

In [4]:
# Thermocouple positions, from the probe axis as recorded and from the probe
# surface as used throughout the analysis.
pos = []
for stage in ('celsio', 'ice2x6', 'clinical', 'crystal_3d'):
    e = next(x for x in EXPS.values() if x['stage'] == stage)
    for ch, r_axis in enumerate(e['tc_pos']):
        role = ('boundary condition' if ch in e['bc_cols'] else
                'offset fit' if ch in e['fit_cols'] else
                'validation' if ch in e['val_cols'] else 'excluded')
        # The registry stores 0.0 for probe-surface channels as a placeholder;
        # physically they sit at the probe radius.
        r = e['r_probe'] if ch in e['bc_cols'] and r_axis == 0.0 else r_axis
        pos.append(dict(stage=stage, channel=ch, role=role,
                        from_axis_mm=r, from_surface_mm=r - e['r_probe']))
positions = pd.DataFrame(pos)
print(positions.to_string(index=False, float_format=lambda v: f'{v:8.2f}'))

     stage  channel               role  from_axis_mm  from_surface_mm
    celsio        0 boundary condition          0.85             0.00
    celsio        1 boundary condition          0.85             0.00
    celsio        2         offset fit          1.80             0.95
    celsio        3         validation          2.10             1.25
    celsio        4         validation          2.40             1.55
    celsio        5           excluded          2.80             1.95
    ice2x6        0 boundary condition          0.75             0.00
    ice2x6        1         offset fit          1.80             1.05
    ice2x6        2         validation          2.80             2.05
    ice2x6        3         validation          4.80             4.05
    ice2x6        4         validation          6.80             6.05
  clinical        0 boundary condition          0.75             0.00
  clinical        1         offset fit          1.80             1.05
  clinical        2 

In [5]:
# Freeze and thaw timings for the in-vitro clinical protocol, from the
# freeze-start sample indices in the thermocouple records at 1 Hz.
FREEZE_S = {1: 180, 2: 420, 3: 600}
timing = []
for rep in ('N1', 'N2', 'N3'):
    fs = {c: EXPS[f'clinical_{rep}_c{c}']['fs'] for c in (1, 2, 3)}
    timing.append(dict(replicate=rep,
                       thaw_1_min=(fs[2] - (fs[1] + FREEZE_S[1])) / 60.0,
                       thaw_2_min=(fs[3] - (fs[2] + FREEZE_S[2])) / 60.0))
timing = pd.DataFrame(timing)
print('In-vitro clinical protocol, thaw durations (min)')
print(timing.to_string(index=False, float_format=lambda v: f'{v:6.2f}'))
print(f"mean second thaw {timing.thaw_2_min.mean():.2f} +/- "
      f"{timing.thaw_2_min.std():.2f} min, against 3 min in the patient protocol")

In-vitro clinical protocol, thaw durations (min)
replicate  thaw_1_min  thaw_2_min
       N1        3.00        4.18
       N2        3.67        3.93
       N3        3.00        3.97
mean second thaw 4.03 +/- 0.14 min, against 3 min in the patient protocol


In [6]:
# Coldest temperature measured at each channel, averaged over experiments. The
# innermost channel not used to set the offset is the coldest independently
# validated temperature; anything colder is model extrapolation.
from cryo_thermal import load_tc, smooth


def measured_minima(exp):
    times, temps = load_tc(exp['file'])
    for ch in range(temps.shape[1]):
        temps[:, ch] = smooth(temps[:, ch])
    for idx, off in exp.get('ch_offset', {}).items():
        if idx < temps.shape[1]:
            temps[:, idx] += off
    win = temps[exp['fs']:min(exp['fs'] + exp['max_dur'], len(times))]
    return {c: float(np.nanmin(win[:, c])) for c in range(temps.shape[1])
            if np.isfinite(win[:, c]).any()}


validated = []
for stage in ('celsio', 'ice2x6', 'clinical', 'crystal_3d'):
    subset = [e for e in EXPS.values() if e['stage'] == stage]
    per_channel = {}
    for e in subset:
        for c, v in measured_minima(e).items():
            per_channel.setdefault(c, []).append(v)
    e0 = subset[0]
    ch = min((c for c in per_channel if c in e0['val_cols']),
             key=lambda c: e0['tc_pos'][c])
    validated.append(dict(stage=stage, channel=ch,
                          from_surface_mm=e0['tc_pos'][ch] - e0['r_probe'],
                          coldest_validated_C=float(np.mean(per_channel[ch])),
                          n=len(per_channel[ch])))
validated = pd.DataFrame(validated)
print(validated.to_string(index=False, float_format=lambda v: f'{v:9.2f}'))

     stage  channel  from_surface_mm  coldest_validated_C  n
    celsio        3             1.25                -2.66 27
    ice2x6        2             2.05               -34.74  6
  clinical        2             2.05               -32.24  9
crystal_3d        2             5.65                 4.54  3


## 3. Dose-response

Per radial bin, `n_expected = max(n_baseline, n_post_total)` and
`death = (n_expected - n_live) / n_expected`, so `n_live` is a count out of
`n_expected`. The observation model is beta-binomial on those counts, with a
three-parameter logistic mean and Gaussian replicate effects.

In [7]:
celsio = pd.read_csv('data/celsio_bins.csv')
ice = pd.read_csv('data/icesphere_bins.csv')
print(f'Celsio rows: {len(celsio)}   IceSphere rows: {len(ice)}')

Celsio rows: 810   IceSphere rows: 360


In [8]:
# Cell lines, temperature domain.
rows_A, groups_A, rep2g_A = dr.build_rows(
    celsio.dropna(subset=['T_min']), x_col='T_min', group_col='cell_line',
    replicate_col='bio_replicate', bin_size=2.0)

# The spread of bin denominators is what makes a single Gaussian variance
# inappropriate: binomial noise on a death fraction scales with 1/sqrt(n).
n = rows_A['n_expected']
print(f'{len(rows_A)} bins, denominators {n.min()} to {n.max()} '
      f'(median {n.median():.0f}), {int((n < 50).sum())} below 50')
print('binomial SD of a death fraction at p = 0.5: '
      f'{100 * np.sqrt(0.25 / 10):.1f} % at n = 10, '
      f'{100 * np.sqrt(0.25 / 1000):.1f} % at n = 1000')

model_A = dr.build_model(rows_A, len(groups_A), rep2g_A, 'betabinomial',
                         domain='temperature')
idata_A = dr.sample(model_A)
print(groups_A)

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`


WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


359 bins, denominators 3 to 3717 (median 862), 6 below 50
binomial SD of a death fraction at p = 0.5: 15.8 % at n = 10, 1.6 % at n = 1000


['A549', 'Calu-1', 'Calu-6']


In [9]:
# Platforms and protocols, temperature domain. The triple-freeze curve is the
# one the clinical projection uses.
_celsio_a549 = (celsio[celsio.cell_line == 'A549']
                .assign(cond='Celsio A549', rep=lambda d: d['bio_replicate']))
_ice = (ice[ice.label.isin(['2x6 F1', 'Clinical'])]
        .assign(cond=lambda d: d['label'].map({'2x6 F1': 'IceSphere F1',
                                               'Clinical': 'IceSphere triple'}),
                rep=lambda d: d['replicate']))
combined = pd.concat([_celsio_a549[['cond', 'rep', 'T_min', 'n_live', 'n_expected']],
                      _ice[['cond', 'rep', 'T_min', 'n_live', 'n_expected']]],
                     ignore_index=True)

rows_B, groups_B, rep2g_B = dr.build_rows(
    combined.dropna(subset=['T_min']), x_col='T_min', group_col='cond',
    replicate_col='rep', bin_size=2.0)

model_B = dr.build_model(rows_B, len(groups_B), rep2g_B, 'betabinomial',
                         domain='temperature')
idata_B = dr.sample(model_B)
print(groups_B)

['Celsio A549', 'IceSphere F1', 'IceSphere triple']


In [10]:
# Convergence.
for name, idata in (('cell lines', idata_A), ('platforms', idata_B)):
    d = dr.diagnostics(idata)
    print(f"{name:11s} max r_hat {d['r_hat'].max():.3f}   "
          f"min bulk ESS {d['ess_bulk'].min():.0f}   "
          f"min tail ESS {d['ess_tail'].min():.0f}   "
          f"max MCSE {d['mcse_mean'].max():.3f}")

cell lines  max r_hat 1.000   min bulk ESS 1707   min tail ESS 1846   max MCSE 0.124
platforms   max r_hat 1.000   min bulk ESS 1557   min tail ESS 2114   max MCSE 0.111


## 4. Choice of observation model

The same mean function and hierarchical structure are refitted under a
Gaussian likelihood on per-bin percentages, a plain binomial on counts, and
the beta-binomial, so the comparison isolates the likelihood. Only the two
count models share an observation space, so only those two are compared by
leave-one-out cross-validation; the Gaussian is judged on interval coverage.

In [11]:
import arviz as az

fits = {'betabinomial': (model_A, idata_A)}
for lik in ('gaussian', 'binomial'):
    m = dr.build_model(rows_A, len(groups_A), rep2g_A, lik, domain='temperature')
    fits[lik] = (m, dr.sample(m))

observed = rows_A['death_pct'].values
cover = {}
for lik, (m, idata) in fits.items():
    pred = dr.posterior_predictive(m, idata, rows_A, lik)
    cover[lik] = dr.interval_coverage(pred, observed)

print('Posterior predictive interval coverage (nominal 0.50 / 0.95)')
for lik in ('gaussian', 'binomial', 'betabinomial'):
    c = cover[lik]
    print(f'  {lik:14s} {c[0.50]:.3f}   {c[0.95]:.3f}')

Sampling: [obs]


Sampling: [obs]


Sampling: [obs]


Posterior predictive interval coverage (nominal 0.50 / 0.95)
  gaussian       0.688   0.942
  binomial       0.128   0.295
  betabinomial   0.549   0.950


In [12]:
# Leave-one-out comparison, count models only.
comp = az.compare({'beta-binomial': fits['betabinomial'][1],
                   'binomial': fits['binomial'][1]}, ic='loo')
print(comp[['rank', 'elpd_loo', 'elpd_diff', 'dse']].to_string())

for lik in ('betabinomial', 'binomial'):
    k = az.loo(fits[lik][1], pointwise=True).pareto_k.values
    print(f'{lik:14s} Pareto k > 0.7: {int((k > 0.7).sum())} of {k.size}, '
          f'max {k.max():.2f}')

C:\Users\arman\AppData\Local\Programs\Python\Python313\Lib\site-packages\arviz\stats\stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


C:\Users\arman\AppData\Local\Programs\Python\Python313\Lib\site-packages\arviz\stats\stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


               rank      elpd_loo    elpd_diff         dse
beta-binomial     0  -1915.222202     0.000000    0.000000
binomial          1 -10643.419078  8728.196876  774.835829


C:\Users\arman\AppData\Local\Programs\Python\Python313\Lib\site-packages\arviz\stats\stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


betabinomial   Pareto k > 0.7: 1 of 359, max 0.73


binomial       Pareto k > 0.7: 68 of 359, max 2.43


C:\Users\arman\AppData\Local\Programs\Python\Python313\Lib\site-packages\arviz\stats\stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


## 5. Dose metrics

The fitted plateau is below 100 % in every condition, so a threshold can be
read relative to that plateau or on an absolute percentage scale. Relative is
used throughout: AD_q = x0 + ln(100/q - 1)/k, so AD50 is the inflection
point by construction and AD95 = x0 - 2.944/k. Absolute values are reported
alongside, with the fraction of posterior draws for which an absolute 95 %
threshold is not attainable.

In [13]:
def dose_table(idata, groups):
    out = []
    post = idata.posterior
    for i, g in enumerate(groups):
        L = post['L'].stack(s=('chain', 'draw')).values[i]
        k = post['k'].stack(s=('chain', 'draw')).values[i]
        rel50 = dr.posterior_dose(idata, i, 50, 'relative')
        rel95 = dr.posterior_dose(idata, i, 95, 'relative')
        abs95 = dr.posterior_dose(idata, i, 95, 'absolute')
        lo50, hi50 = dr.hdi_of(rel50)
        lo95, hi95 = dr.hdi_of(rel95)
        out.append(dict(
            group=g, L=L.mean(), k=k.mean(),
            AD50=np.nanmean(rel50), AD50_lo=lo50, AD50_hi=hi50,
            AD95=np.nanmean(rel95), AD95_lo=lo95, AD95_hi=hi95,
            AD95_absolute=(np.nanmean(abs95) if np.isfinite(abs95).any() else np.nan),
            pct_absolute_undefined=100.0 * np.mean(~np.isfinite(abs95))))
    return pd.DataFrame(out)


tab_A = dose_table(idata_A, groups_A)
tab_B = dose_table(idata_B, groups_B)
print('Cell lines (degC)')
print(tab_A.to_string(index=False, float_format=lambda v: f'{v:8.2f}'))
print('\nPlatforms and protocols (degC)')
print(tab_B.to_string(index=False, float_format=lambda v: f'{v:8.2f}'))

Cell lines (degC)
 group        L        k     AD50  AD50_lo  AD50_hi     AD95  AD95_lo  AD95_hi  AD95_absolute  pct_absolute_undefined
  A549    89.07     0.12   -23.34   -30.11   -16.85   -47.17   -55.12   -39.93            NaN                  100.00
Calu-1    94.04     0.13    -6.38   -16.65     3.83   -29.69   -40.41   -18.78         -57.22                   89.18
Calu-6    95.93     0.13   -12.95   -20.81    -6.25   -35.40   -43.22   -27.62         -49.22                    5.98

Platforms and protocols (degC)
           group        L        k     AD50  AD50_lo  AD50_hi     AD95  AD95_lo  AD95_hi  AD95_absolute  pct_absolute_undefined
     Celsio A549    89.39     0.13   -23.40   -30.27   -17.18   -45.63   -53.04   -38.45            NaN                  100.00
    IceSphere F1    95.10     0.09    -8.47   -13.95    -3.58   -41.19   -48.58   -34.35         -66.82                   44.78
IceSphere triple    96.58     0.20    -9.77   -18.37    -1.13   -24.47   -33.15   -15.69      

In [14]:
# Directional contrasts.
def p_less(idata, groups, a, b, var='x0'):
    v = idata.posterior[var].stack(s=('chain', 'draw')).values
    return float(np.mean(v[groups.index(a)] < v[groups.index(b)]))


print('P(AD50 A549 < Calu-1)      ', f"{p_less(idata_A, groups_A, 'A549', 'Calu-1'):.3f}")
print('P(AD50 A549 < Calu-6)      ', f"{p_less(idata_A, groups_A, 'A549', 'Calu-6'):.3f}")
print('P(AD50 Celsio < IceSphere) ',
      f"{p_less(idata_B, groups_B, 'Celsio A549', 'IceSphere F1'):.3f}")
print('P(plateau Celsio < IceSphere)',
      f"{p_less(idata_B, groups_B, 'Celsio A549', 'IceSphere F1', var='L'):.3f}")

P(AD50 A549 < Calu-1)       0.994
P(AD50 A549 < Calu-6)       0.981
P(AD50 Celsio < IceSphere)  0.999
P(plateau Celsio < IceSphere) 1.000


## 6. Multi-cycle survival

If each freeze-thaw cycle acts as an independent trial in which surviving
cells face the same probability of lethal ice formation, cumulative survival
after two cycles is the square of survival after one. Both are measured
against the same pre-ablation cell density, bin by bin, so the exponent in
S2 = S1**gamma can be estimated directly from the counts.

In [15]:
from scipy.optimize import minimize_scalar
from scipy.stats import binom, t as student_t

f1 = ice[ice.label == '2x6 F1'].set_index(['replicate', 'dist_from_surface_mm'])
f2 = ice[ice.label == '2x6 F2'].set_index(['replicate', 'dist_from_surface_mm'])
surv = (f1[['n_live', 'n_expected']]
        .join(f2[['n_live', 'n_expected']], lsuffix='_1', rsuffix='_2').dropna())
surv['S1'] = surv['n_live_1'] / surv['n_expected_1']
surv['S2'] = surv['n_live_2'] / surv['n_expected_2']


def survival_exponent(sub):
    """Binomial maximum likelihood estimate of gamma in S2 = S1**gamma."""
    S1 = sub['S1'].values.clip(1e-6, 1 - 1e-6)
    k = sub['n_live_2'].values.astype(int)
    n = sub['n_expected_2'].values.astype(int)

    def nll(g):
        return -np.sum(binom.logpmf(k, n, np.clip(S1 ** g, 1e-9, 1 - 1e-9)))

    return float(minimize_scalar(nll, bounds=(0.1, 8.0), method='bounded').x)


# Saturated bins next to the probe carry no information about the exponent.
usable = (surv['S1'] > 0.05) & (surv['S1'] < 0.95) & (surv['n_expected_2'] >= 20)
gammas = {rep: survival_exponent(surv.loc[rep]) for rep in sorted(
    surv.index.get_level_values(0).unique())}
g = np.array(list(gammas.values()))
ci = student_t.ppf(0.975, len(g) - 1) * g.std(ddof=1) / np.sqrt(len(g))

print('per replicate: ' + ', '.join(f'{r} {v:.2f}' for r, v in gammas.items()))
print(f'pooled over {int(usable.sum())} bins: gamma = {survival_exponent(surv[usable]):.2f}')
print(f'across replicates: gamma = {g.mean():.2f} +/- {g.std(ddof=1):.2f} (SD), '
      f'95% CI [{g.mean() - ci:.2f}, {g.mean() + ci:.2f}]')

per replicate: N1 2.94, N2 2.42, N3 2.08
pooled over 102 bins: gamma = 2.42
across replicates: gamma = 2.48 +/- 0.43 (SD), 95% CI [1.41, 3.55]


## 7. Lung-adapted geometry

The solver is reparameterised with lung tissue properties and run through the
clinical three-cycle protocol. The radial extent at each contour is mapped to
a confocal prolate ellipsoid whose foci sit at the ends of the 22 mm active
freezing length.

In [16]:
import lung_chain

runs = [r for r in (lung_chain.chain_protocol(rep, 'VC3', -90.0)
                    for rep in ('N1', 'N2', 'N3')) if r is not None]

AD50_C = float(tab_B.loc[tab_B.group == 'IceSphere triple', 'AD50'].iloc[0])
AD95_C = float(tab_B.loc[tab_B.group == 'IceSphere triple', 'AD95'].iloc[0])

rows = []
for label, T in (('ice-ball edge (0 C)', 0.0), ('-20 C', -20.0), ('-40 C', -40.0),
                 ('AD50', AD50_C), ('AD95', AD95_C)):
    r_tip = np.mean([dr.find_isotherm_dist(r['T_min_field'], r['d_grid'], T)
                     for r in runs])
    geo = dr.tip_to_ellipsoid(r_tip)
    rows.append(dict(contour=label, T_C=T, r_tip_mm=r_tip,
                     length_mm=geo['length'], width_mm=geo['width_max'],
                     volume_mm3=geo['volume']))
geom = pd.DataFrame(rows)
print(geom.to_string(index=False, float_format=lambda v: f'{v:9.2f}'))

v = dict(zip(geom.contour, geom.volume_mm3))
print(f"\nAD95 volume / -40 C volume = {v['AD95'] / v['-40 C']:.3f}")

            contour       T_C  r_tip_mm  length_mm  width_mm  volume_mm3
ice-ball edge (0 C)      0.00      5.00      27.56     16.61     3980.21
              -20 C    -20.00      2.83      25.01     11.89     1852.63
              -40 C    -40.00      1.53      23.58      8.49      889.50
               AD50     -9.77      3.78      26.10     14.04     2695.11
               AD95    -24.47      2.48      24.62     11.06     1576.17

AD95 volume / -40 C volume = 1.772


## 8. Agreement with the patient cohort

`data/cohort_geometry.csv` holds the mean and covariance of the measured
orthogonal ice-ball diameters in the 52-patient cohort. These summary
statistics reproduce the Mahalanobis distance and the equivalence test;
per-patient coverage requires the individual measurements, which are
available from the corresponding author on request.

In [17]:
from scipy.stats import chi2

coh = pd.read_csv('data/cohort_geometry.csv').set_index('quantity')
n_pat = int(coh.loc['n', 'long_axis'])
cohort_mean = coh.loc['mean_mm', ['long_axis', 'short_axis']].values.astype(float)
cohort_sd = coh.loc['sd_mm', ['long_axis', 'short_axis']].values.astype(float)
cohort_cov = np.array([coh.loc['cov_long', ['long_axis', 'short_axis']].values,
                       coh.loc['cov_short', ['long_axis', 'short_axis']].values],
                      dtype=float)

model_xy = geom.loc[geom.contour == 'ice-ball edge (0 C)',
                    ['length_mm', 'width_mm']].values[0]
diff_vec = model_xy - cohort_mean
d2 = float(diff_vec @ np.linalg.inv(cohort_cov) @ diff_vec)
print(f'model {model_xy.round(2)}   cohort {cohort_mean.round(2)}')
print(f'Mahalanobis d2 = {d2:.3f},  chi2_2 P = {chi2.sf(d2, 2):.3f}')

# Two one-sided tests. The smallest margin at which equivalence holds is the
# larger absolute bound of the 90 % confidence interval on the difference.
se = cohort_sd / np.sqrt(n_pat)
tcrit = student_t.ppf(0.95, n_pat - 1)
for i, axis in enumerate(('long', 'short')):
    d = cohort_mean[i] - model_xy[i]
    lo, hi = d - tcrit * se[i], d + tcrit * se[i]
    print(f'{axis:5s} axis  difference {d:+6.3f} mm   90% CI [{lo:+.3f}, {hi:+.3f}]   '
          f'smallest equivalence margin {max(abs(lo), abs(hi)):.3f} mm')

model [27.56 16.61]   cohort [25.35 15.98]
Mahalanobis d2 = 0.614,  chi2_2 P = 0.736
long  axis  difference -2.214 mm   90% CI [-2.958, -1.470]   smallest equivalence margin 2.958 mm
short axis  difference -0.628 mm   90% CI [-1.298, +0.042]   smallest equivalence margin 1.298 mm


## 9. Modified Stefan number

In [18]:
for medium, T_med, T_probe, label, T_fus, L_f in (
        ('PBS (Celsio)', 30.74, -79.0, 'CO2', -0.52, ct.L_WATER),
        ('PBS (IceSphere)', 27.27, -184.0, 'argon', -0.52, ct.L_WATER),
        ('GelMA (3D)', 29.23, -184.0, 'argon', -2.8, ct.L_GELMA)):
    st = dr.stefan_number(T_probe=T_probe, T_medium=T_med, T_fusion=T_fus,
                          c_p_solid=ct.C_ICE, c_p_liquid=ct.C_WATER, L_f=L_f)
    print(f'{medium:18s} {label:6s} T_M = {T_med:5.2f} C   St* = {st:.3f}')

PBS (Celsio)       CO2    T_M = 30.74 C   St* = 0.353
PBS (IceSphere)    argon  T_M = 27.27 C   St* = 0.852
GelMA (3D)         argon  T_M = 29.23 C   St* = 0.906
